In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from xgboost import XGBClassifier

project_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_path / "data"
model_path = project_path / "models"

model_path.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path / "dataset.csv")

if "label" not in df.columns:
    raise ValueError("The dataset must contain a label column.")

if df["label"].isna().any():
    raise ValueError("Target labels contain missing values.")

if "patient_id" in df.columns:
    df = df.drop(columns=["patient_id"])

X = df.drop(columns=["label"])
y = df["label"].astype(int)

if y.nunique() != 2 or not set(y.unique()).issubset({0, 1}):
    raise ValueError("This notebook expects binary labels encoded as 0 and 1.")

feature_columns = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=250,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

results = []

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    scores = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    }

    results.append(scores)

    filename = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(model, model_path / filename)

    print(pd.Series(scores).to_string())
    print("\nClassification report:")
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

results_df = pd.DataFrame(results).sort_values(
    "F1 Score", ascending=False
).reset_index(drop=True)

print("\nMODEL COMPARISON")
display(results_df.round(4))

results_df.to_csv(
    data_path / "model_comparison.csv",
    index=False
)

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

joblib.dump(best_model, model_path / "best_model.joblib")
joblib.dump(feature_columns, model_path / "feature_columns.joblib")

print("\nBest model based on F1 Score:", best_model_name)
print("Saved model:", model_path / "best_model.joblib")
print("Saved feature columns:", model_path / "feature_columns.joblib")
print("Saved comparison:", data_path / "model_comparison.csv")

print("\nModel training completed.")



Training Logistic Regression...
Model        Logistic Regression
Accuracy                    0.68
Precision                 0.6875
Recall                      0.66
F1 Score                0.673469
ROC-AUC                 0.737524

Classification report:
              precision    recall  f1-score   support

           0       0.67      0.70      0.69       500
           1       0.69      0.66      0.67       500

    accuracy                           0.68      1000
   macro avg       0.68      0.68      0.68      1000
weighted avg       0.68      0.68      0.68      1000


Training Random Forest...
Model        Random Forest
Accuracy             0.654
Precision         0.659751
Recall               0.636
F1 Score          0.647658
ROC-AUC            0.70986

Classification report:
              precision    recall  f1-score   support

           0       0.65      0.67      0.66       500
           1       0.66      0.64      0.65       500

    accuracy                           0.

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.680,0.6875,0.660,0.6735,0.7375
1,XGBoost,0.672,0.6713,0.674,0.6727,0.7209
2,Random Forest,0.654,0.6598,0.636,0.6477,0.7099



Best model based on F1 Score: Logistic Regression
Saved model: c:\Users\saura\OneDrive\Desktop\early_disease_prediction\models\best_model.joblib
Saved feature columns: c:\Users\saura\OneDrive\Desktop\early_disease_prediction\models\feature_columns.joblib
Saved comparison: c:\Users\saura\OneDrive\Desktop\early_disease_prediction\data\model_comparison.csv

Model training completed.
